In [ ]:
import os, glob, hashlib, json, logging, re
from pathlib import Path
from tempfile import TemporaryDirectory
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
logging.disable(logging.CRITICAL)
try:
    assert not os.path.exists('/dev/nvidia0')
    import yaml, pyarrow.parquet as pq
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import snapshot_download
    sources = glob.glob('/kaggle/input/**/xbrl-frontier/docs/config/thresholds.yaml', recursive=True)
    assert len(sources) == 1
    pin = yaml.safe_load(Path(sources[0]).read_text())['pins']['eval_finni']
    try:
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        detail = str(exc)
        if not re.fullmatch(r'No user secrets exist for kernel id [0-9]+ and label HF_TOKEN', detail):
            detail = '[redacted]'
        print(json.dumps({'status': 'D4_ACCESS_FAILED', 'error_type': type(exc).__name__, 'detail': detail[:120]}))
        raise SystemExit(1) from None
    assert token
    with TemporaryDirectory(dir='/tmp') as cache:
        root = Path(snapshot_download(repo_id=pin['repo'], repo_type='dataset', revision=pin['hf_revision'], cache_dir=cache, token=token))
        shards = sorted(root.glob('data/' + pin['split'] + '-*.parquet'))
        assert len(shards) == 1
        rows = sum(pq.read_metadata(path).num_rows for path in shards)
        with shards[0].open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        assert rows == pin['rows_expected']
        assert digest == 'd13a93783e256b75c743fe4743dde1de1e9a5e80c972a137afcb783c81d73dcb'
    del token
    print(json.dumps({'status': 'D4_ACCESS_OK', 'revision': pin['hf_revision'], 'rows': rows, 'shard_sha256': digest}))
except Exception as exc:
    print(json.dumps({'status': 'D4_ACCESS_FAILED', 'error_type': type(exc).__name__}))
    raise SystemExit(1) from None
